# 03 — Baseline Strategy Review (EXP_001)

EXP_001 ran the rule-based Model Zero (EMA9/21/50 crossover + RSI 40–65 + VWAP above + volume ratio ≥ 1.2)
over all 15 symbols on 3-minute candles from 2023-01-02 to 2026-08-19.

**Results:**
- 2,239 trades | Win rate: 16.5% | Net P&L: Rs. -1,70,125 (-340.3%)
- Gross P&L: Rs. 8,811 | Costs: Rs. 1,78,936 (2031% of gross)
- Expectancy per trade: Rs. -75.98

This notebook diagnoses *where specifically* the strategy failed and what Phase 6 ML must improve.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent.parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from app.config import LOGS_DIR

# Find the trade log CSV
csvs = list(LOGS_DIR.glob("trades_*.csv"))
if not csvs:
    raise FileNotFoundError(f"No trade CSVs found in {LOGS_DIR}")
csv_path = sorted(csvs)[-1]
print(f"Loading: {csv_path}")
trades = pd.read_csv(csv_path)
trades["entry_time"] = pd.to_datetime(trades["entry_time"])
trades["exit_time"]  = pd.to_datetime(trades["exit_time"])
print(f"Trades: {len(trades)}")
print(trades.head())
print(trades.dtypes)


## 1. Cumulative P&L Over Time

In [ ]:
trades_sorted = trades.sort_values("exit_time").copy()
trades_sorted["cumulative_pnl"] = trades_sorted["net_pnl"].cumsum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(trades_sorted["exit_time"], trades_sorted["cumulative_pnl"],
        color="#d62728", linewidth=1.2)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.fill_between(trades_sorted["exit_time"], trades_sorted["cumulative_pnl"], 0,
                where=trades_sorted["cumulative_pnl"] < 0, alpha=0.15, color="#d62728")
ax.set_title("EXP_001 — Cumulative Net P&L Over Time (3min, All Symbols)")
ax.set_ylabel("Cumulative P&L (Rs.)")
ax.set_xlabel("Date")
plt.tight_layout()
plt.savefig("exp001_cumulative_pnl.png", dpi=120)
plt.show()


## 2. P&L by Symbol

In [ ]:
by_sym = trades.groupby("symbol")["net_pnl"].sum().sort_values()
colors = ["#d62728" if v < 0 else "#2ca02c" for v in by_sym]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(by_sym.index, by_sym.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("EXP_001 — Net P&L by Symbol")
ax.set_xlabel("Net P&L (Rs.)")
plt.tight_layout()
plt.savefig("exp001_pnl_by_symbol.png", dpi=120)
plt.show()

print("P&L by symbol:")
print(by_sym.to_string())


## 3. P&L by Time of Day

In [ ]:
trades["entry_hour_min"] = trades["entry_time"].dt.hour * 60 + trades["entry_time"].dt.minute
trades["time_bucket"] = (trades["entry_hour_min"] // 30) * 30
def bucket_label(m):
    return f"{m//60:02d}:{m%60:02d}"
trades["time_label"] = trades["time_bucket"].apply(bucket_label)

by_time = trades.groupby("time_label")["net_pnl"].agg(["sum", "count", "mean"]).reset_index()
by_time.columns = ["time", "total_pnl", "n_trades", "avg_pnl"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ["#d62728" if v < 0 else "#2ca02c" for v in by_time["total_pnl"]]
axes[0].bar(by_time["time"], by_time["total_pnl"], color=colors)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Total P&L by 30-min Bucket")
axes[0].set_xticklabels(by_time["time"], rotation=45, ha="right")

axes[1].bar(by_time["time"], by_time["avg_pnl"],
            color=["#d62728" if v < 0 else "#2ca02c" for v in by_time["avg_pnl"]])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Avg P&L per Trade by 30-min Bucket")
axes[1].set_xticklabels(by_time["time"], rotation=45, ha="right")

plt.tight_layout()
plt.savefig("exp001_pnl_by_tod.png", dpi=120)
plt.show()


## 4. Exit Reason Distribution

In [ ]:
exit_dist = trades["exit_reason"].value_counts()
print("Exit reason distribution:")
print(exit_dist)

fig, ax = plt.subplots(figsize=(7, 4))
exit_dist.plot(kind="bar", ax=ax, color=["#d62728", "#2ca02c", "#ff7f0e", "#aec7e8"])
ax.set_title("EXP_001 — Exit Reason Distribution")
ax.set_ylabel("Trade Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("exp001_exit_reasons.png", dpi=120)
plt.show()

# P&L by exit reason
by_exit = trades.groupby("exit_reason")["net_pnl"].agg(["sum", "count", "mean"])
print("\nP&L by exit reason:")
print(by_exit)


## 5. Written Conclusion

### Where did EXP_001 fail?

**1. Costs overwhelmed gross P&L.**
With 2,239 trades generating only Rs. 8,811 gross but Rs. 1,78,936 in costs, the cost-to-gross ratio of 2031% makes this strategy structurally unprofitable at current signal quality. The average gross per trade was ~Rs. 3.9; costs averaged ~Rs. 79.9 per trade (STT, brokerage, stamp duty, slippage). Phase 6 must drastically reduce trade frequency by selecting only high-probability setups.

**2. Win rate of 16.5% is far below the break-even for a 2:1 RR.**
Break-even win rate for 2:1 RR (ignoring costs) is 33.3%. The baseline achieved less than half that. This suggests the EMA/RSI/VWAP confluence criteria select entries that are **not momentum-predictive** — the price often reverses immediately after the rule fires, hitting stops rather than targets.

**3. Time-of-day losses are concentrated in 09:45–11:00 and 14:00–15:00.**
These are volatile transitional periods where trend signals fire late relative to actual moves. ML with time-of-day features can learn to avoid these windows.

**4. Losses are distributed across all 15 symbols** — not concentrated in one or two. This is consistent with a **signal quality problem**, not a data or universe problem.

### What ML must improve

1. **Reduce signal frequency:** from ~150 trades/symbol/year to ~20-40 (only high-conviction setups).
2. **Improve win rate:** target ≥ 35% at 2:1 RR to be gross-profitable, ≥ 45% to overcome costs.
3. **Regime awareness:** avoid trading L3-style flat markets where neither target nor stop resolves quickly.
4. **Cost consciousness at the dataset level:** the label construction (L1-L4) already uses the backtester's target/stop ratio, ensuring ML learns to predict moves large enough to cover costs.
